# Proxy evals (Omni-MATH + LAB-Bench LitQA2) on Colab GPU
Own-benchmark numbers only. Never E-lane scores.

In [ ]:
!pip -q install datasets transformers accelerate bitsandbytes

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"  # change as needed
N_OMNI, N_LIT = 20, 50

In [ ]:
from datasets import load_dataset
omni = load_dataset("KbsdJames/Omni-MATH", split=f"test[:{N_OMNI}]")
lit = load_dataset("futurehouse/lab-bench", "LitQA2", split=f"train[:{N_LIT}]")
print(len(omni), len(lit), omni.column_names, lit.column_names)

In [ ]:
import torch
from transformers import pipeline
gen = pipeline("text-generation", model=MODEL_ID, model_kwargs={"dtype": torch.bfloat16},
               device_map="auto", max_new_tokens=256)
print(gen("What is 2+2? Reply with just the number.", return_full_text=False)[0]["generated_text"][:50])

In [ ]:
import re, json
def norm(s):
    return re.sub(r"\s+", "", re.sub(r"\\(boxed|text|mathrm)\{?", "", s.strip()))
res = []
for i, row in enumerate(omni):
    out = gen("Solve this math problem. Put your final answer on its own last line starting with 'FINAL: '\n\n" + row["problem"], return_full_text=False)[0]["generated_text"]
    m = re.findall(r"FINAL:\s*(.+)", out)
    pred = norm(m[-1]) if m else ""
    gold = norm(row["answer"])
    ok = bool(pred) and (pred == gold or pred in gold or gold in pred)
    res.append({"i": i, "correct": bool(ok)})
    print(i, bool(ok))
acc = sum(r["correct"] for r in res)/len(res)
print("Omni-MATH accuracy:", acc)
json.dump({"proxy":"Omni-MATH-test-subset-colab","model":MODEL_ID,"n":len(res),"accuracy":acc,"results":res}, open("OMNIMATH_COLAB.json","w"), indent=1)

In [ ]:
import ast, random
random.seed(211)
res2 = []
for i, row in enumerate(lit):
    try: distractors = ast.literal_eval(row["distractors"])
    except Exception: continue
    opts = distractors + [row["ideal"]]
    random.shuffle(opts)
    letters = "ABCD"
    prompt = row["question"] + "\n" + "\n".join(f"{L}. {o}" for L, o in zip(letters, opts)) + "\nAnswer with only the letter."
    out = gen(prompt, return_full_text=False)[0]["generated_text"]
    m = re.search(r"\b([A-D])\b", out)
    pred = opts[letters.index(m.group(1))] if m else ""
    ok = pred == row["ideal"]
    res2.append({"id": row["id"], "correct": bool(ok)})
    print(i, bool(ok))
acc2 = sum(r["correct"] for r in res2)/max(1,len(res2))
print("LitQA2 accuracy:", acc2)
json.dump({"proxy":"LAB-Bench-LitQA2-subset-colab","model":MODEL_ID,"n":len(res2),"accuracy":acc2,"results":res2}, open("LABBENCH_COLAB.json","w"), indent=1)